# TetraFT — Phase 1 / 1b smoke (0.8B on Kaggle)

**Attach datasets**
- `tetraft-code` — flat `.py` modules
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** (T4/P100-class) |
| Internet | **ON** first run (download Qwen); need recent `transformers` for `qwen3_5` |
| Flow | inventory → original PPL → shock PPL → QAFT |

**Presets** (`run_smoke.py --preset`)
| Preset | Steps | ≈ tokens (1×512×8 = 4096/step) |
|--------|------:|--------------------------------:|
| `short` | 200 | ~0.82M (Phase 1 baseline — done) |
| `longer` | 640 | ~2.6M (Phase 1b) |
| `full_smoke` | 1280 | ~5.2M |

Logic lives in `run_smoke.py` — this notebook is glue only.

In [ ]:
# Qwen3.5 needs a recent transformers (model_type qwen3_5).
# If stock Kaggle fails with KeyError qwen3_5, install from source:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

In [ ]:
from run_smoke import run_smoke
import argparse

# Phase 1 baseline used preset="short" (done on Kaggle).
# Phase 1b: set PRESET = "longer" (~2.6M tok) or "full_smoke" (~5.2M).
PRESET = "longer"

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir="/kaggle/working/checkpoints",
    seq_length=None,       # preset / config default
    batch_size=None,
    max_steps=None,        # from preset unless overridden
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=False,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    seed=42,
    device_map="auto",
)
results = run_smoke(ns)
keys = ["preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
        "loss_finite", "tokens_seen", "tokens_budget", "steps_ran"]
print({k: results[k] for k in keys if k in results})

Artifacts under `/kaggle/working/checkpoints/`:
- `linear_inventory.json`
- `smoke_results.json` (`tokens_seen`, PPL curve in `train_metrics`)
- `checkpoint-*`

**Phase 1 baseline (short, Kaggle):** orig PPL ≈ 17.7, shock ≫ 1e6, ~200 steps → eval PPL ≈ 472, loss finite.

**Phase 1b success:** PPL trend down vs ~472 at λ=1; still compare gap to original (~17.7), not BitNet.